# Music Visualizer - Image Analyzer
Prepare an image for the Music Visualizer in three steps:

1. **Load** an image and choose the output folder.
2. **Colours** - tune the colour ranges, then confirm to export the colour layers and glow.
3. **Structure** - tune the dark-outline detection, then confirm to export the structure and area masks.

This notebook is designed to run as an app with [Voila](https://voila.readthedocs.io): `python -m voila ImageAnalyzer_App.ipynb`.

In [ ]:
import os
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Defaults are the tuned values. Red wraps around the OpenCV hue circle (0-179), so it has two ranges.
DEFAULT_COLOR_RANGES = {
    'red': [([0, 80, 80], [10, 255, 255]), ([160, 80, 80], [179, 255, 255])],
    'orange': [([10, 80, 80], [20, 255, 255])],
    'yellow': [([20, 20, 20], [35, 255, 255])],
    'green': [([35, 30, 30], [85, 255, 255])],
    'cyan': [([85, 20, 20], [100, 255, 255])],
    'blue': [([100, 30, 30], [140, 255, 255])],
    'magenta': [([140, 20, 20], [160, 255, 255])],
}
DEFAULT_DARK = dict(value_threshold=65, chroma_threshold=35, kernel_size=3, iterations=1, min_area=12)
PREVIEW_COLORS_RGB = {
    'red': (255, 0, 0), 'orange': (255, 140, 0), 'yellow': (255, 255, 0), 'green': (0, 200, 0),
    'cyan': (0, 255, 255), 'blue': (0, 0, 255), 'magenta': (255, 0, 255),
}
DEFAULT_OUTPUT_DIR = os.environ.get('IMAGE_ANALYZER_OUTPUT', 'Visualizer/data')   # the bundle launchers set this
MAX_OUTPUT_DIMENSION = 1920   # Processing does not need full-size textures
PREVIEW_DIMENSION = 700
BLOOM_KERNEL = 151

state = {}   # data of the currently loaded image


def decode_image(data):
    """Decode image bytes into a BGRA uint8 array, keeping the alpha channel when present."""
    img = cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError('The file could not be read as an image.')
    if img.dtype != np.uint8:
        img = (img.astype(np.float64) * (255.0 / np.iinfo(img.dtype).max)).astype(np.uint8) \
            if np.issubdtype(img.dtype, np.integer) else np.clip(img * 255, 0, 255).astype(np.uint8)
    if img.ndim == 2:
        return cv2.cvtColor(img, cv2.COLOR_GRAY2BGRA)
    if img.shape[2] == 3:   # no alpha: fully opaque
        return cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
    if img.shape[2] == 4:
        return img
    raise ValueError(f'Unsupported image shape: {img.shape}')


def prepare_image(img_bgra):
    """Derive every working copy of the image needed by the three steps."""
    img_bgr = img_bgra[:, :, :3]
    height, width = img_bgr.shape[:2]
    scale = min(1.0, MAX_OUTPUT_DIMENSION / max(width, height))
    output_size = (max(1, round(width * scale)), max(1, round(height * scale)))
    preview_scale = min(1.0, PREVIEW_DIMENSION / max(width, height))
    preview_size = (max(1, round(width * preview_scale)), max(1, round(height * preview_scale)))

    if scale == 1.0:
        structure_bgr, structure_alpha = img_bgr.copy(), img_bgra[:, :, 3].copy()
    else:
        structure_bgr = cv2.resize(img_bgr, output_size, interpolation=cv2.INTER_AREA)
        structure_alpha = cv2.resize(img_bgra[:, :, 3], output_size, interpolation=cv2.INTER_NEAREST)

    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    state.clear()
    state.update(
        bgra=img_bgra,
        alpha=img_bgra[:, :, 3],
        hsv=img_hsv,
        rgb=cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
        output_scale=scale,
        output_size=output_size,
        preview_hsv=cv2.resize(img_hsv, preview_size, interpolation=cv2.INTER_AREA),
        preview_alpha=cv2.resize(img_bgra[:, :, 3], preview_size, interpolation=cv2.INTER_NEAREST),
        preview_rgb=cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), preview_size, interpolation=cv2.INTER_AREA),
        structure_value=cv2.cvtColor(structure_bgr, cv2.COLOR_BGR2HSV)[:, :, 2],
        structure_chroma=structure_bgr.max(axis=2) - structure_bgr.min(axis=2),
        structure_visible=structure_alpha > 0,
    )


def resize_for_processing(image, interpolation=cv2.INTER_AREA):
    if state['output_scale'] == 1.0:
        return image
    return cv2.resize(image, state['output_size'], interpolation=interpolation)


def build_color_masks(hsv, alpha, color_ranges):
    masks = {}
    for name, ranges in color_ranges.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lower, upper in ranges:
            mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.array(lower, dtype=np.uint8), np.array(upper, dtype=np.uint8)))
        # Exclude transparent pixels, including hidden RGB data under alpha.
        masks[name] = cv2.bitwise_and(mask, alpha)
    return masks


def add_residual(masks, alpha):
    """Whatever no colour mask covers (greys, blacks, whites)."""
    covered = np.zeros_like(alpha)
    for mask in masks.values():
        covered = cv2.bitwise_or(covered, mask)
    masks['residual'] = cv2.bitwise_and(alpha, cv2.bitwise_not(covered))
    return masks


def compute_dark_mask(value_threshold, chroma_threshold, kernel_size, iterations, min_area):
    """Cleaned near-black structure mask (255 = structure) and the number of removed specks."""
    # Low value finds dark pixels; low chroma keeps dark saturated glass from becoming frame.
    candidates = (
        (state['structure_value'] <= value_threshold)
        & (state['structure_chroma'] <= chroma_threshold)
        & state['structure_visible']
    ).astype(np.uint8) * 255
    if iterations > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        candidates = cv2.morphologyEx(candidates, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    _, labels, stats, _ = cv2.connectedComponentsWithStats(candidates, connectivity=8)
    keep = stats[:, cv2.CC_STAT_AREA] >= min_area
    keep[0] = False   # background
    return np.where(keep[labels], 255, 0).astype(np.uint8), int(np.count_nonzero(~keep[1:]))


def export_color_layers(output_dir, color_ranges, report=lambda done, total, name: None):
    os.makedirs(output_dir, exist_ok=True)
    cv2.imwrite(os.path.join(output_dir, 'window_base.png'), resize_for_processing(state['bgra']))
    masks = add_residual(build_color_masks(state['hsv'], state['alpha'], color_ranges), state['alpha'])
    rgb = state['rgb']
    for index, (name, mask) in enumerate(masks.items(), start=1):
        layer = np.zeros((rgb.shape[0], rgb.shape[1], 4), dtype=np.uint8)
        layer[mask > 0, :3] = rgb[mask > 0]
        layer[mask > 0, 3] = mask[mask > 0]
        # Blur at source resolution, then downsample only the exported textures.
        bloom = cv2.GaussianBlur(layer, (BLOOM_KERNEL, BLOOM_KERNEL), 0)
        cv2.imwrite(os.path.join(output_dir, f'mask_{name}_solid.png'),
                    cv2.cvtColor(resize_for_processing(layer), cv2.COLOR_RGBA2BGRA))
        cv2.imwrite(os.path.join(output_dir, f'mask_{name}_bloom.png'),
                    cv2.cvtColor(resize_for_processing(bloom), cv2.COLOR_RGBA2BGRA))
        report(index, len(masks), name)
    return list(masks)


def encode_area_ids(labels):
    """Pack integer area IDs into the B, G, R channels of a standard PNG."""
    ids = np.zeros((*labels.shape, 3), dtype=np.uint8)
    ids[:, :, 0] = labels & 0xFF
    ids[:, :, 1] = (labels >> 8) & 0xFF
    ids[:, :, 2] = (labels >> 16) & 0xFF
    return ids


def export_structure(output_dir, dark_params):
    os.makedirs(output_dir, exist_ok=True)
    cleaned, removed = compute_dark_mask(**dark_params)
    shape_mask = cv2.bitwise_not(cleaned)
    cv2.imwrite(os.path.join(output_dir, 'mask_colored_areas.png'), shape_mask)
    count, labels = cv2.connectedComponents(shape_mask, connectivity=4)
    cv2.imwrite(os.path.join(output_dir, 'mask_area_ids.png'), encode_area_ids(labels))
    return shape_mask, labels, count, removed


def to_png(rgb):
    ok, buffer = cv2.imencode('.png', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    return buffer.tobytes()

In [ ]:
def hint(text):
    return widgets.HTML(f'<span style="color:#666">{text}</span>')

def status_html(text, kind='info'):
    color = {'info': '#333', 'ok': '#1a7f37', 'error': '#cf222e'}[kind]
    return f'<div style="color:{color};margin:6px 0">{text}</div>'


# ---------- Step 1: load ----------
upload = widgets.FileUpload(accept='image/*,.png,.jpg,.jpeg,.bmp,.tif,.tiff,.webp', multiple=False,
                            description='Choose image...', button_style='primary')
output_dir_input = widgets.Text(value=DEFAULT_OUTPUT_DIR, description='Output folder',
                                style={'description_width': '110px'}, layout=widgets.Layout(width='600px'))
browse_button = widgets.Button(description='Browse...', icon='folder-open')
load_status = widgets.HTML()

FOLDER_DIALOG_SCRIPT = (
    'import sys, tkinter as tk\n'
    'from tkinter import filedialog\n'
    'root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)\n'
    'print(filedialog.askdirectory(title="Select the output folder", initialdir=sys.argv[1], mustexist=False))\n'
)

def choose_output_folder(_=None):
    """Open a native folder dialog on this machine. It runs in a subprocess because tkinter needs a main thread."""
    import subprocess, sys
    start = os.path.abspath(output_dir_input.value.strip() or DEFAULT_OUTPUT_DIR)
    while start and not os.path.isdir(start):
        parent = os.path.dirname(start)
        if parent == start:
            break
        start = parent
    browse_button.disabled = True
    try:
        result = subprocess.run([sys.executable, '-c', FOLDER_DIALOG_SCRIPT, start],
                                capture_output=True, text=True, timeout=600)
        chosen = result.stdout.strip()
        if result.returncode != 0:
            load_status.value = status_html('The folder dialog is not available here; please type the folder path instead.', 'error')
        elif chosen:
            output_dir_input.value = os.path.normpath(chosen)
    except Exception:
        load_status.value = status_html('The folder dialog is not available here; please type the folder path instead.', 'error')
    finally:
        browse_button.disabled = False

browse_button.on_click(choose_output_folder)
original_view = widgets.Image(format='png')


def first_upload(value):
    """Return (name, bytes) for the uploaded file; supports ipywidgets 7 and 8."""
    if not value:
        return None
    item = value[0] if isinstance(value, (tuple, list)) else next(iter(value.values()))
    if isinstance(value, dict) and not isinstance(value, (tuple, list)):
        return next(iter(value.keys())), item['content']
    return item['name'], bytes(item['content'])


# ---------- Step 2: colours ----------
def range_slider(low, high, maximum, label):
    return widgets.IntRangeSlider(value=[low, high], min=0, max=maximum, description=label,
                                  continuous_update=False, layout=widgets.Layout(width='380px'))

color_range_widgets, color_rows = {}, []
for color_name, ranges in DEFAULT_COLOR_RANGES.items():
    color_range_widgets[color_name] = []
    for index, (lower, upper) in enumerate(ranges):
        suffix = f' #{index + 1}' if len(ranges) > 1 else ''
        triplet = (range_slider(lower[0], upper[0], 179, 'H'),
                   range_slider(lower[1], upper[1], 255, 'S'),
                   range_slider(lower[2], upper[2], 255, 'V'))
        color_range_widgets[color_name].append(triplet)
        color_rows.append(widgets.HBox([widgets.HTML(f'<b>{color_name.capitalize()}{suffix}</b>',
                                                     layout=widgets.Layout(width='90px')), *triplet]))

def read_color_ranges():
    return {name: [([h.value[0], s.value[0], v.value[0]], [h.value[1], s.value[1], v.value[1]])
                   for h, s, v in triplets] for name, triplets in color_range_widgets.items()}

def reset_color_ranges(_=None):
    for name, triplets in color_range_widgets.items():
        for (h, s, v), (lower, upper) in zip(triplets, DEFAULT_COLOR_RANGES[name]):
            h.value, s.value, v.value = [lower[0], upper[0]], [lower[1], upper[1]], [lower[2], upper[2]]

color_preview = widgets.Image(format='png')
color_stats = widgets.HTML()
reset_colors_button = widgets.Button(description='Reset to defaults', icon='undo')
confirm_colors_button = widgets.Button(description='Confirm and export colour layers', button_style='success', icon='check')
color_progress = widgets.IntProgress(min=0, max=1, value=0, description='Export', layout=widgets.Layout(width='400px'))
color_status = widgets.HTML()

def update_color_preview(_=None):
    if not state:
        return
    masks = build_color_masks(state['preview_hsv'], state['preview_alpha'], read_color_ranges())
    canvas = np.zeros((*state['preview_hsv'].shape[:2], 3), dtype=np.uint8)
    covered = np.zeros(canvas.shape[:2], dtype=bool)
    for name, mask in masks.items():
        canvas[mask > 0] = PREVIEW_COLORS_RGB[name]
        covered |= mask > 0
    residual = (state['preview_alpha'] > 0) & ~covered
    canvas[residual] = (128, 128, 128)
    color_preview.value = to_png(np.hstack([state['preview_rgb'], canvas]))
    visible = max(1, np.count_nonzero(state['preview_alpha']))
    parts = ', '.join(f'{n}: {100 * np.count_nonzero(m) / visible:.1f}%' for n, m in masks.items())
    color_stats.value = hint(f'Left: original. Right: colour classes (grey = residual).<br>'
                             f'Coverage - {parts}, residual: {100 * np.count_nonzero(residual) / visible:.1f}%').value

for triplets in color_range_widgets.values():
    for triplet in triplets:
        for w in triplet:
            w.observe(update_color_preview, names='value')
reset_colors_button.on_click(reset_color_ranges)


# ---------- Step 3: structure ----------
def int_slider(default, minimum, maximum, label):
    return widgets.IntSlider(value=default, min=minimum, max=maximum, description=label, continuous_update=False,
                             style={'description_width': '230px'}, layout=widgets.Layout(width='600px'))

dark_widgets = {
    'value_threshold': int_slider(DEFAULT_DARK['value_threshold'], 0, 255, 'Max brightness (value)'),
    'chroma_threshold': int_slider(DEFAULT_DARK['chroma_threshold'], 0, 255, 'Max colourfulness (chroma)'),
    'kernel_size': int_slider(DEFAULT_DARK['kernel_size'], 1, 15, 'Gap closing kernel size'),
    'iterations': int_slider(DEFAULT_DARK['iterations'], 0, 5, 'Gap closing iterations'),
    'min_area': int_slider(DEFAULT_DARK['min_area'], 0, 500, 'Min dark component area (px)'),
}
def read_dark_params():
    return {name: w.value for name, w in dark_widgets.items()}

def reset_dark_params(_=None):
    for name, w in dark_widgets.items():
        w.value = DEFAULT_DARK[name]

structure_preview = widgets.Image(format='png')
structure_info = widgets.HTML()
reset_dark_button = widgets.Button(description='Reset to defaults', icon='undo')
confirm_structure_button = widgets.Button(description='Confirm and export structure masks', button_style='success', icon='check')
structure_status = widgets.HTML()

def update_structure_preview(_=None):
    if not state:
        return
    cleaned, removed = compute_dark_mask(**read_dark_params())
    structure_preview.value = to_png(cv2.cvtColor(cv2.bitwise_not(cleaned), cv2.COLOR_GRAY2RGB))
    structure_info.value = hint(f'Inverted near-black mask preview. Tiny dark components removed: {removed}').value

for w in dark_widgets.values():
    w.observe(update_structure_preview, names='value')
reset_dark_button.on_click(reset_dark_params)


# ---------- Layout and wiring ----------
step2 = widgets.VBox([widgets.HTML('<h3>2. Colours</h3>'),
                      hint('HSV ranges per colour (OpenCV hue is 0-179, saturation and value 0-255).'),
                      *color_rows, widgets.HBox([reset_colors_button]), color_stats, color_preview,
                      widgets.HBox([confirm_colors_button, color_progress]), color_status])
step3 = widgets.VBox([widgets.HTML('<h3>3. Structure</h3>'),
                      hint('Detect near-black, low-chroma outlines, reconnect small gaps and drop tiny specks.'),
                      *dark_widgets.values(), widgets.HBox([reset_dark_button]), structure_info, structure_preview,
                      confirm_structure_button, structure_status])
step2.layout.display = 'none'
step3.layout.display = 'none'


def on_upload(change):
    try:
        picked = first_upload(upload.value)
        if picked is None:
            return
        name, data = picked
        load_status.value = status_html(f'Loading {name}...')
        prepare_image(decode_image(data))
        height, width = state['alpha'].shape
        load_status.value = status_html(
            f'Loaded <b>{name}</b>: {width} x {height} px. Export size: {state["output_size"][0]} x {state["output_size"][1]} px.', 'ok')
        original_view.value = to_png(state['preview_rgb'])
        color_status.value = structure_status.value = ''
        step3.layout.display = 'none'
        step2.layout.display = ''
        update_color_preview()
    except Exception as error:
        state.clear()
        step2.layout.display = step3.layout.display = 'none'
        load_status.value = status_html(f'Could not load the image: {error}', 'error')


def on_confirm_colors(_):
    confirm_colors_button.disabled = True
    try:
        out = output_dir_input.value.strip() or DEFAULT_OUTPUT_DIR
        color_status.value = status_html('Exporting colour layers and glow (this can take a minute)...')
        def report(done, total, name):
            color_progress.max, color_progress.value = total, done
            color_status.value = status_html(f'Exported {name} ({done}/{total})')
        export_color_layers(out, read_color_ranges(), report)
        color_status.value = status_html(f'Colour layers saved to <code>{os.path.abspath(out)}</code>. Continue with step 3.', 'ok')
        step3.layout.display = ''
        update_structure_preview()
    except Exception as error:
        color_status.value = status_html(f'Export failed: {error}', 'error')
    finally:
        confirm_colors_button.disabled = False


def on_confirm_structure(_):
    confirm_structure_button.disabled = True
    try:
        out = output_dir_input.value.strip() or DEFAULT_OUTPUT_DIR
        structure_status.value = status_html('Exporting structure masks...')
        _, _, count, removed = export_structure(out, read_dark_params())
        structure_status.value = status_html(
            f'Done. Saved <code>mask_colored_areas.png</code> and <code>mask_area_ids.png</code> ({count} areas, '
            f'{removed} specks removed) to <code>{os.path.abspath(out)}</code>. The Visualizer can now use these files.', 'ok')
    except Exception as error:
        structure_status.value = status_html(f'Export failed: {error}', 'error')
    finally:
        confirm_structure_button.disabled = False


upload.observe(on_upload, names='value')
confirm_colors_button.on_click(on_confirm_colors)
confirm_structure_button.on_click(on_confirm_structure)

display(widgets.VBox([
    widgets.HTML('<h3>1. Load image</h3>'),
    widgets.HBox([upload]), widgets.HBox([output_dir_input, browse_button]), load_status, original_view,
    step2, step3,
]))